In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("robot_arm_forward_kinematics.ipynb")

# Robot arm

forward kinematics: Building the robot arm and forward kinematics

inverse kinmeatics: Make the arm go somewhere (robot_arm_inverse_kinematics)

Slides: https://docs.google.com/presentation/d/1r7Aja2nI4iaLmaFbhugOf9mnopdR3ZFnkyVOKvLqKZc/edit?usp=sharing

In [ ]:
# The usual imports
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import fmin

In [ ]:
# matrix_rouintes.py functions
import matrix_routines as mt

# These are the classes you wrote/are writing in arm_geometry_practice
from arm_component import ArmComponent
from robot_arm import RobotArm2D
from gripper import Gripper

In [ ]:
# These commands will force JN to actually re-load the external file when you re-execute the import command
%load_ext autoreload
%autoreload 2

# Part 1: Building an arm and doing forward kinematics

Two parts to this problem: The first is creating all of the components and storing them in a class (rather than having each one named individually, as was done in the practice JN). This way, you can have an arbitrary number of links in the arm.

The second part of this assignment is computing the pose matrices for each link, given the angles for each link/finger/gripper.

GUIDES: Fill in robot_arm.py and gripper.py. 

## Part 1: Put all the components together

Step 1: Create the link components based on a list of link/base sizes. This is the **__init__** method in **robot_arm.py**

Step 2: Fill in the **get_** methods in **robot_arm.py**

In [ ]:
# These are example inputs for the create_arm_geometry function. 
base_size_param = (0.5, 1.0)   # Height, width
link_sizes_param = [(0.5, 0.25), (0.3, 0.1), (0.2, 0.05)]

# The sizes for all of the components
palm_size = 0.1
finger_length = 0.075

robot_arm = RobotArm2D(base_arm_size=base_size_param, link_sizes=link_sizes_param, 
                       palm_size=palm_size, finger_length=finger_length)

In [ ]:
# Correct shape matrices
mat_base_check = np.array([[0.5, 0.0, 0], [0.0, 0.25, 0.25], [0.0, 0.0, 1.0]])
mat_link1_check = np.array([[0.25, 0.0, 0.25], [0.0, 0.125, 0.0], [0.0, 0.0, 1.0]])
mat_link2_check = np.array([[0.15, 0.0, 0.15], [0.0, 0.05, 0.0], [0.0, 0.0, 1.0]])
mat_link3_check = np.array([[0.1, 0.0, 0.1], [0.0, 0.025, 0.0], [0.0, 0.0, 1.0]])

# Check the gripper get method and the shape matrices
# Correct shape matrices
mat_palm_check = np.array([[0.005, 0.0, 0.0], [0.0, 0.05, 0.0], [0.0, 0.0, 1.0]])
mat_finger_top_check = np.array([[0.0, 0.0375, 0.0375], [-0.0125, 0.0, 0.05], [0.0, 0.0, 1.0]])
mat_finger_bot_check = np.array([[0.0, 0.0375, 0.0375], [-0.0125, 0.0, -0.05], [0.0, 0.0, 1.0]])

# Example check 
assert np.all(np.isclose(robot_arm.get_gripper().get_palm_shape_matrix(), mat_palm_check))


In [ ]:
# This checks the base shape matrix. Notice:
#   Using get_base to get the instance of the base matrix
#   Calling get_shape_matrix() on that instance to get the shape matrix
#   Doing the comparison using np.all(np.isclose())
assert np.all(np.isclose(robot_arm.get_base().get_shape_matrix(), mat_base_check))

# GUIDES: Use the get methods and the "check" matrices above to check the rest of your matrices and your get functions
# Note: The autograder calls these checks, once for the base, once for each link, and once each for the palm & fingers. 

In [ ]:
grader.check("build_arm")

### Part 2: Forward kinematics

In this step you'll fill in **set_link_angles**, which will set the pose matrices of every link **ArmComponent**.

(See slides) Each component has a "chain" of matrices that takes the link to the right location. This chain consists of alternating rotation and translation matrices. Each chain consists of all of the translations and rotations for the previous links plus a translation and rotation matrix for the current link. The gripper location has the longest chain - it consists of all of the rotations and translation for every link. 

One confusing thing is that the *last* matrix to be applied is the first one in the chain, reading from left to right in the code (see slides for why). 

There is one "special" matrix - this is the one that takes the first link to the top of the base, pointed up. This is the *last* matrix in the chain. 

Building the chain is the same for all of the links, so you can do this in a **for** loop. This loop is a bit tricky because you don't *quite* take the matrix from the previous link and just add the next rotation/translation pair in the chain. So your code will look like:

**matrix_chain =** special base matrix

- For the link, add a rotation by the link's angle then a translation in x by the **previous** link's length

- Add this rotation/translation pair to the chain in **matrix_chain**

If the **for** loop is challenging, just set each matrix in turn and then look for the pattern.

For full credit this has to work no matter what the link lengths are and how many links there are (hence the **for** loop)

Reminder that you should be using the **set_pose_matrix** method to set the matrices for the components (so it plots properly).

Second reminder: You should be using mt.make_xx_matrix routines to make the matrices.

Third reminder: You can store information (like link length) in the arm component class.

In [ ]:
# Now set the arm angles and the gripper angles
angles_check = [np.pi/2, -np.pi/4, -3.0 * np.pi/4]
palm_angle = np.pi/3.0
finger_angle = np.pi/4.0
robot_arm.set_link_angles(link_angles=angles_check, palm_angle=palm_angle, finger_angle=finger_angle)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(6, 3))
robot_arm.set_link_angles(link_angles=angles_check, palm_angle=palm_angle, finger_angle=0.0)
robot_arm.plot(axs=axs[0])
axs[0].set_title("No fingers")

robot_arm.set_link_angles(link_angles=angles_check, palm_angle=palm_angle, finger_angle=finger_angle)
robot_arm.plot(axs=axs[1])
axs[1].set_title("Fingers")

fig.tight_layout()
plt.show()


In [ ]:
mat_check_base = np.identity(3)
mat_check_link_1 = np.array([[ -1.0,  0.0,  0.0], \
                             [  0.0, -1.0,  0.5], \
                             [  0.0,  0.0,  1.0]])
                             
mat_check_link_2 = np.array([[ -0.7071, -0.7071, -0.5], \
                             [  0.7071, -0.7071,  0.5], \
                             [  0.0,     0.0,  1.0]])

mat_check_link_3 = np.array([[ 1.0, 0.0, -0.71213], \
                             [ 0.0, 1.0,  0.71213], \
                             [ 0.0, 0.0,  1.0]])

for ilink, m in enumerate((mat_check_link_1, mat_check_link_2, mat_check_link_3)):
    assert(np.all(np.isclose(robot_arm.get_link(ilink).get_pose_matrix(), m, atol=0.01)))

mat_check_wrist = np.array([[ 0.5, -0.8660,  -0.5121], \
                            [ 0.8660,  0.5,   0.7121], \
                            [ 0.0,  0.0,  1.0]])
print(robot_arm.get_gripper().get_palm_pose_matrix())
assert np.all(np.isclose(robot_arm.get_gripper().get_palm_pose_matrix(), mat_check_wrist, atol=0.01))

mat_check_f1 = np.array([[-0.25881, -0.965926,  -0.507137], \
                         [ 0.965926, -0.258819,   0.750073], \
                         [ 0.0,  0.0,  1.0]])
print(robot_arm.get_gripper().get_finger_pose_matrix(is_top=True))
assert np.all(np.isclose(robot_arm.get_gripper().get_finger_pose_matrix(is_top=True), mat_check_f1, atol=0.01))

mat_check_f2 = np.array([[0.965926, -0.258819,  -0.481772], \
                         [0.258819, 0.965926,   0.735428], \
                         [ 0.0,  0.0,  1.0]])
print(robot_arm.get_gripper().get_finger_pose_matrix(is_top=False))
assert np.all(np.isclose(robot_arm.get_gripper().get_finger_pose_matrix(is_top=False), mat_check_f2, atol=0.01))


In [ ]:
grader.check("forward_ik")

# Gripper location

GUIDES: edit **get_grasp_location** in gripper.py to return the x,y location of the gripper.


In [ ]:
# Check the end location method
# Do not change these, tho, because the autograder won't work
angles_check_end_location = [np.pi/3, -np.pi/6, 3.0 * np.pi/6]
grasp_percentage = 0.75

robot_arm_2 = RobotArm2D(base_arm_size=base_size_param, link_sizes=link_sizes_param, palm_size=palm_size, finger_length=finger_length, grasp_percentage=grasp_percentage)

# As in the previous problem, you can use the "simpler" angles to check your function
robot_arm_2.set_link_angles(angles_check_end_location, palm_angle=palm_angle)

# Check the end location is correct (there is plotting code in the next cell)
end_loc = robot_arm_2.get_gripper().get_grasp_location()
assert np.isclose(end_loc[0], -0.7562, atol=0.01) and np.isclose(end_loc[1], 0.8536, atol=0.01)

In [ ]:

# Now actually plot - the grasp grip location should show up as a green cross
fig2, axs2 = plt.subplots(1, 1, figsize=(6, 6))
robot_arm_2.plot(axs2)


In [ ]:
# Do NOT change these or the autograder tests won't work
angles_check_end_location = [np.pi/3, -np.pi/6, 3.0 * np.pi/6]

# Actually set the matrices
robot_arm_2.set_link_angles(angles_check_end_location, palm_angle=palm_angle)

# Check the end location is correct (there is plotting code in the next cell)
end_loc_check = robot_arm_2.get_gripper().get_grasp_location()
print(f"{end_loc_check}")


In [ ]:
grader.check("gripper_loc")

## Generalization check

If nothing has been "hardwired" in, this should just work - changing the geometry, the starting angles, the target point. However, if you've hardwired something in, it probably won't...

In [ ]:
# Create another arm geometry
base_size_longer_param = (0.5, 0.25) # squished (height, width)
link_sizes_longer_param = [(0.3, 0.15), (0.2, 0.09), (0.1, 0.05), (0.075, 0.03)]
palm_width_longer_param = 0.15
finger_length_longer_param = (0.085, 0.015)


# Ths creates a different robot arm - one that is longer and with an additinal link
robot_arm_longer = RobotArm2D(base_size_longer_param, 
                                link_sizes_longer_param, 
                                palm_size=palm_size, 
                                finger_length=finger_length, 
                                grasp_percentage=0.5)

# Set the angles of the arm
angles_start_longer = [-np.pi/4.0, -np.pi/4, 1.2 * np.pi/4, -1 * np.pi/8]
robot_arm_longer.set_link_angles(angles_start_longer, palm_angle=-0.2, finger_angle=0.3)

end_loc_longer_check = robot_arm_longer.get_gripper().get_grasp_location()


assert np.all(np.isclose(end_loc_longer_check[0:2], (0.56683, 0.85181), atol=0.01))

In [ ]:
# Now actually plot - the grasp grip location should show up as a green cross
fig3, axs3 = plt.subplots(1, 1, figsize=(6, 6))
robot_arm_longer.plot(axs3)
axs3.set_title("Longer arm")

In [ ]:
grader.check("generalization_check")

## Hours and collaborators
Required for every assignment - fill out before you hand-in.

Listing names and websites helps you to document who you worked with and what internet help you received in the case of any plagiarism issues. You should list names of anyone (in class or not) who has substantially helped you with an assignment - or anyone you have *helped*. You do not need to list TAs.

Listing hours helps us track if the assignments are too long.

In [ ]:

# List of names (creates a set)
worked_with_names = {"not filled out"}
# List of URLS FA26 (creates a set)
websites = {"not filled out"}
# Approximate number of hours, including lab/in-class time
hours = -1.5

In [ ]:
grader.check("hours_collaborators")

### To submit


- Submit this .ipynb file AND arm_component.py AND robot_arm.py AND gripper.py. If you don't include the .py files Gradescope cannot magically reach out to your computer and find it.
- We will supply matrix_routines.py for you (it won't break anything if you do include it).
- As always, do a restart-runall before submitting and make sure your plots are visible.

If the Gradescope autograder fails, please check here first for common reasons for it to fail
    https://docs.google.com/presentation/d/1tYa5oycUiG4YhXUq5vHvPOpWJ4k_xUPp2rUNIL7Q9RI/edit?usp=sharing

Lots of people forget the .py files. Please check your autograder score to see if you are one of them.

Make sure you remove all the print statements you put in that print out lots of stuff.